# v3: choosing the best model with repeated stratified 5-fold CV

**Same data and goals as v1 and v2.** The 480 measured values are never changed, removed or synthesised; this is checked at the end.

**What is new in v3**
1. **Repeated stratified grouped 5-fold cross-validation** (5 folds × 3 repeats = 15 evaluations) is used to **choose** the best model.
   - *Grouped:* all replicates of a run, and all 5 materials at that run, stay in the same fold.
   - *Stratified:* each fold gets a fair share of every design block (centre, the four one-factor blocks, factorial, extreme corners).
2. **ANN convergence check:** how many iterations each network actually needs, compared with the iteration limit.
3. **New ANN variants:** a 2-layer ANN, and an Adam-trained ANN with early stopping.
4. The chosen model is then **confirmed** on the untouched hold-out test runs and on leave-one-condition-out. Choosing (5-fold CV) and confirming (hold-out, LOCO) use different measures, so the reported accuracy is not flattered by the selection.

Candidates: everything from v1 and v2, plus the two new ANNs.

In [ ]:
import io, os, zipfile, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm, statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy import stats
from scipy.optimize import minimize
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200, 'display.max_columns', 30)
SEED = 42
OUT = 'results'; os.makedirs(OUT, exist_ok=True)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.ensemble import HistGradientBoostingRegressor
from joblib import Parallel, delayed

In [ ]:
# @title Lab data (hidden: 480 measured rows, nothing added)
CSV = """mat,run,conc,time,pH,dose,x1,x2,x3,x4,rep,y
Biochar,1,50,180,8,50,0,0,0,0,R1,-1.960784
Biochar,1,50,180,8,50,0,0,0,0,R2,-2.941176
Biochar,1,50,180,8,50,0,0,0,0,R3,-1.960784
Biochar,2,50,180,8,50,0,0,0,0,R1,-3.186275
Biochar,2,50,180,8,50,0,0,0,0,R2,-1.22549
Biochar,2,50,180,8,50,0,0,0,0,R3,-1.22549
Biochar,3,50,180,8,50,0,0,0,0,R1,-1.470588
Biochar,3,50,180,8,50,0,0,0,0,R2,-1.960784
Biochar,3,50,180,8,50,0,0,0,0,R3,-2.205882
Biochar,4,50,180,8,50,0,0,0,0,R1,-0.980392
Biochar,4,50,180,8,50,0,0,0,0,R2,0.245098
Biochar,4,50,180,8,50,0,0,0,0,R3,-0.735294
Biochar,5,25,180,8,50,-1,0,0,0,R1,-0.245098
Biochar,5,25,180,8,50,-1,0,0,0,R2,-1.22549
Biochar,5,25,180,8,50,-1,0,0,0,R3,-0.735294
Biochar,6,75,180,8,50,1,0,0,0,R1,0.980392
Biochar,6,75,180,8,50,1,0,0,0,R2,2.45098
Biochar,6,75,180,8,50,1,0,0,0,R3,1.715686
Biochar,7,10,180,8,50,-2,0,0,0,R1,-0.784314
Biochar,7,10,180,8,50,-2,0,0,0,R2,-1.642157
Biochar,7,10,180,8,50,-2,0,0,0,R3,-1.151961
Biochar,8,100,180,8,50,2,0,0,0,R1,2.205882
Biochar,8,100,180,8,50,2,0,0,0,R2,3.431373
Biochar,8,100,180,8,50,2,0,0,0,R3,1.715686
Biochar,9,50,120,8,50,0,-1,0,0,R1,-0.245098
Biochar,9,50,120,8,50,0,-1,0,0,R2,-0.245098
Biochar,9,50,120,8,50,0,-1,0,0,R3,-0.245098
Biochar,10,50,240,8,50,0,1,0,0,R1,-1.715686
Biochar,10,50,240,8,50,0,1,0,0,R2,-2.45098
Biochar,10,50,240,8,50,0,1,0,0,R3,-2.205882
Biochar,11,50,60,8,50,0,-2,0,0,R1,-1.470588
Biochar,11,50,60,8,50,0,-2,0,0,R2,-2.205882
Biochar,11,50,60,8,50,0,-2,0,0,R3,-2.205882
Biochar,12,50,360,8,50,0,2,0,0,R1,-2.941176
Biochar,12,50,360,8,50,0,2,0,0,R2,-1.715686
Biochar,12,50,360,8,50,0,2,0,0,R3,-2.696078
Biochar,13,50,180,6,50,0,0,-1,0,R1,-0.245098
Biochar,13,50,180,6,50,0,0,-1,0,R2,-2.696078
Biochar,13,50,180,6,50,0,0,-1,0,R3,-2.205882
Biochar,14,50,180,10,50,0,0,1,0,R1,0.245098
Biochar,14,50,180,10,50,0,0,1,0,R2,-0.490196
Biochar,14,50,180,10,50,0,0,1,0,R3,0.245098
Biochar,15,50,180,4,50,0,0,-2,0,R1,-1.960784
Biochar,15,50,180,4,50,0,0,-2,0,R2,-1.715686
Biochar,15,50,180,4,50,0,0,-2,0,R3,-1.22549
Biochar,16,50,180,12,50,0,0,2,0,R1,0
Biochar,16,50,180,12,50,0,0,2,0,R2,0
Biochar,16,50,180,12,50,0,0,2,0,R3,0.245098
Biochar,17,50,180,8,20,0,0,0,-1,R1,-5.514706
Biochar,17,50,180,8,20,0,0,0,-1,R2,-3.063725
Biochar,17,50,180,8,20,0,0,0,-1,R3,-4.901961
Biochar,18,50,180,8,100,0,0,0,1,R1,0.122549
Biochar,18,50,180,8,100,0,0,0,1,R2,-0.367647
Biochar,18,50,180,8,100,0,0,0,1,R3,-0.490196
Biochar,19,50,180,8,10,0,0,0,-2,R1,-2.45098
Biochar,19,50,180,8,10,0,0,0,-2,R2,-11.029412
Biochar,19,50,180,8,10,0,0,0,-2,R3,-7.352941
Biochar,20,50,180,8,150,0,0,0,2,R1,-0.163399
Biochar,20,50,180,8,150,0,0,0,2,R2,0.163399
Biochar,20,50,180,8,150,0,0,0,2,R3,-0.081699
Biochar,21,25,120,6,20,-1,-1,-1,-1,R1,-2.144608
Biochar,21,25,120,6,20,-1,-1,-1,-1,R2,-5.514706
Biochar,21,25,120,6,20,-1,-1,-1,-1,R3,-4.289216
Biochar,22,25,240,10,100,-1,1,1,1,R1,-1.041667
Biochar,22,25,240,10,100,-1,1,1,1,R2,-1.102941
Biochar,22,25,240,10,100,-1,1,1,1,R3,-0.735294
Biochar,23,75,120,10,20,1,-1,1,-1,R1,-14.093137
Biochar,23,75,120,10,20,1,-1,1,-1,R2,-13.480392
Biochar,23,75,120,10,20,1,-1,1,-1,R3,-11.642157
Biochar,24,75,240,6,100,1,1,-1,1,R1,-0.612745
Biochar,24,75,240,6,100,1,1,-1,1,R2,0.612745
Biochar,24,75,240,6,100,1,1,-1,1,R3,0.367647
Biochar,25,25,120,10,100,-1,-1,1,1,R1,0.061275
Biochar,25,25,120,10,100,-1,-1,1,1,R2,-0.122549
Biochar,25,25,120,10,100,-1,-1,1,1,R3,-0.061275
Biochar,26,25,240,6,20,-1,1,-1,-1,R1,-3.370098
Biochar,26,25,240,6,20,-1,1,-1,-1,R2,-0.306373
Biochar,26,25,240,6,20,-1,1,-1,-1,R3,-1.838235
Biochar,27,75,120,6,100,1,-1,-1,1,R1,2.083333
Biochar,27,75,120,6,100,1,-1,-1,1,R2,1.348039
Biochar,27,75,120,6,100,1,-1,-1,1,R3,1.715686
Biochar,28,75,240,10,20,1,1,1,-1,R1,-3.063725
Biochar,28,75,240,10,20,1,1,1,-1,R2,-0.612745
Biochar,28,75,240,10,20,1,1,1,-1,R3,-3.676471
Biochar,29,10,60,4,10,-2,-2,-2,-2,R1,-9.436275
Biochar,29,10,60,4,10,-2,-2,-2,-2,R2,-10.04902
Biochar,29,10,60,4,10,-2,-2,-2,-2,R3,-10.04902
Biochar,30,10,360,12,150,-2,2,2,2,R1,0.555556
Biochar,30,10,360,12,150,-2,2,2,2,R2,0.02451
Biochar,30,10,360,12,150,-2,2,2,2,R3,0.392157
Biochar,31,100,60,12,10,2,-2,2,-2,R1,-11.029412
Biochar,31,100,60,12,10,2,-2,2,-2,R2,-17.156863
Biochar,31,100,60,12,10,2,-2,2,-2,R3,-14.705882
Biochar,32,100,360,4,150,2,2,-2,2,R1,4.166667
Biochar,32,100,360,4,150,2,2,-2,2,R2,4.411765
Biochar,32,100,360,4,150,2,2,-2,2,R3,3.921569
Repyrolysis 1:1,1,50,180,8,50,0,0,0,0,R1,35.294118
Repyrolysis 1:1,1,50,180,8,50,0,0,0,0,R2,35.539216
Repyrolysis 1:1,1,50,180,8,50,0,0,0,0,R3,35.294118
Repyrolysis 1:1,2,50,180,8,50,0,0,0,0,R1,34.313725
Repyrolysis 1:1,2,50,180,8,50,0,0,0,0,R2,35.784314
Repyrolysis 1:1,2,50,180,8,50,0,0,0,0,R3,34.558824
Repyrolysis 1:1,3,50,180,8,50,0,0,0,0,R1,33.823529
Repyrolysis 1:1,3,50,180,8,50,0,0,0,0,R2,34.558824
Repyrolysis 1:1,3,50,180,8,50,0,0,0,0,R3,33.333333
Repyrolysis 1:1,4,50,180,8,50,0,0,0,0,R1,35.04902
Repyrolysis 1:1,4,50,180,8,50,0,0,0,0,R2,33.823529
Repyrolysis 1:1,4,50,180,8,50,0,0,0,0,R3,36.029412
Repyrolysis 1:1,5,25,180,8,50,-1,0,0,0,R1,23.406863
Repyrolysis 1:1,5,25,180,8,50,-1,0,0,0,R2,22.794118
Repyrolysis 1:1,5,25,180,8,50,-1,0,0,0,R3,22.54902
Repyrolysis 1:1,6,75,180,8,50,1,0,0,0,R1,40.196078
Repyrolysis 1:1,6,75,180,8,50,1,0,0,0,R2,36.27451
Repyrolysis 1:1,6,75,180,8,50,1,0,0,0,R3,37.745098
Repyrolysis 1:1,7,10,180,8,50,-2,0,0,0,R1,9.509804
Repyrolysis 1:1,7,10,180,8,50,-2,0,0,0,R2,9.387255
Repyrolysis 1:1,7,10,180,8,50,-2,0,0,0,R3,9.632353
Repyrolysis 1:1,8,100,180,8,50,2,0,0,0,R1,54.656863
Repyrolysis 1:1,8,100,180,8,50,2,0,0,0,R2,49.754902
Repyrolysis 1:1,8,100,180,8,50,2,0,0,0,R3,48.284314
Repyrolysis 1:1,9,50,120,8,50,0,-1,0,0,R1,27.696078
Repyrolysis 1:1,9,50,120,8,50,0,-1,0,0,R2,28.431373
Repyrolysis 1:1,9,50,120,8,50,0,-1,0,0,R3,29.166667
Repyrolysis 1:1,10,50,240,8,50,0,1,0,0,R1,35.784314
Repyrolysis 1:1,10,50,240,8,50,0,1,0,0,R2,36.764706
Repyrolysis 1:1,10,50,240,8,50,0,1,0,0,R3,34.558824
Repyrolysis 1:1,11,50,60,8,50,0,-2,0,0,R1,22.794118
Repyrolysis 1:1,11,50,60,8,50,0,-2,0,0,R2,21.813725
Repyrolysis 1:1,11,50,60,8,50,0,-2,0,0,R3,21.078431
Repyrolysis 1:1,12,50,360,8,50,0,2,0,0,R1,35.784314
Repyrolysis 1:1,12,50,360,8,50,0,2,0,0,R2,36.27451
Repyrolysis 1:1,12,50,360,8,50,0,2,0,0,R3,37.009804
Repyrolysis 1:1,13,50,180,6,50,0,0,-1,0,R1,40.196078
Repyrolysis 1:1,13,50,180,6,50,0,0,-1,0,R2,40.686275
Repyrolysis 1:1,13,50,180,6,50,0,0,-1,0,R3,39.460784
Repyrolysis 1:1,14,50,180,10,50,0,0,1,0,R1,32.843137
Repyrolysis 1:1,14,50,180,10,50,0,0,1,0,R2,34.558824
Repyrolysis 1:1,14,50,180,10,50,0,0,1,0,R3,35.294118
Repyrolysis 1:1,15,50,180,4,50,0,0,-2,0,R1,44.852941
Repyrolysis 1:1,15,50,180,4,50,0,0,-2,0,R2,45.098039
Repyrolysis 1:1,15,50,180,4,50,0,0,-2,0,R3,45.833333
Repyrolysis 1:1,16,50,180,12,50,0,0,2,0,R1,21.568627
Repyrolysis 1:1,16,50,180,12,50,0,0,2,0,R2,21.078431
Repyrolysis 1:1,16,50,180,12,50,0,0,2,0,R3,21.323529
Repyrolysis 1:1,17,50,180,8,20,0,0,0,-1,R1,37.377451
Repyrolysis 1:1,17,50,180,8,20,0,0,0,-1,R2,39.828431
Repyrolysis 1:1,17,50,180,8,20,0,0,0,-1,R3,41.053922
Repyrolysis 1:1,18,50,180,8,100,0,0,0,1,R1,24.264706
Repyrolysis 1:1,18,50,180,8,100,0,0,0,1,R2,23.406863
Repyrolysis 1:1,18,50,180,8,100,0,0,0,1,R3,24.019608
Repyrolysis 1:1,19,50,180,8,10,0,0,0,-2,R1,42.892157
Repyrolysis 1:1,19,50,180,8,10,0,0,0,-2,R2,42.892157
Repyrolysis 1:1,19,50,180,8,10,0,0,0,-2,R3,44.117647
Repyrolysis 1:1,20,50,180,8,150,0,0,0,2,R1,16.503268
Repyrolysis 1:1,20,50,180,8,150,0,0,0,2,R2,16.584967
Repyrolysis 1:1,20,50,180,8,150,0,0,0,2,R3,16.094771
Repyrolysis 1:1,21,25,120,6,20,-1,-1,-1,-1,R1,33.394608
Repyrolysis 1:1,21,25,120,6,20,-1,-1,-1,-1,R2,27.879902
Repyrolysis 1:1,21,25,120,6,20,-1,-1,-1,-1,R3,28.79902
Repyrolysis 1:1,22,25,240,10,100,-1,1,1,1,R1,11.029412
Repyrolysis 1:1,22,25,240,10,100,-1,1,1,1,R2,10.968137
Repyrolysis 1:1,22,25,240,10,100,-1,1,1,1,R3,11.151961
Repyrolysis 1:1,23,75,120,10,20,1,-1,1,-1,R1,56.372549
Repyrolysis 1:1,23,75,120,10,20,1,-1,1,-1,R2,42.279412
Repyrolysis 1:1,23,75,120,10,20,1,-1,1,-1,R3,53.308824
Repyrolysis 1:1,24,75,240,6,100,1,1,-1,1,R1,34.791667
Repyrolysis 1:1,24,75,240,6,100,1,1,-1,1,R2,36.25
Repyrolysis 1:1,24,75,240,6,100,1,1,-1,1,R3,35.729167
Repyrolysis 1:1,25,25,120,10,100,-1,-1,1,1,R1,10.171569
Repyrolysis 1:1,25,25,120,10,100,-1,-1,1,1,R2,10.416667
Repyrolysis 1:1,25,25,120,10,100,-1,-1,1,1,R3,9.375
Repyrolysis 1:1,26,25,240,6,20,-1,1,-1,-1,R1,53.308824
Repyrolysis 1:1,26,25,240,6,20,-1,1,-1,-1,R2,53.308824
Repyrolysis 1:1,26,25,240,6,20,-1,1,-1,-1,R3,56.678922
Repyrolysis 1:1,27,75,120,6,100,1,-1,-1,1,R1,32.107843
Repyrolysis 1:1,27,75,120,6,100,1,-1,-1,1,R2,34.926471
Repyrolysis 1:1,27,75,120,6,100,1,-1,-1,1,R3,34.068627
Repyrolysis 1:1,28,75,240,10,20,1,1,1,-1,R1,36.151961
Repyrolysis 1:1,28,75,240,10,20,1,1,1,-1,R2,41.053922
Repyrolysis 1:1,28,75,240,10,20,1,1,1,-1,R3,39.215686
Repyrolysis 1:1,29,10,60,4,10,-2,-2,-2,-2,R1,46.323529
Repyrolysis 1:1,29,10,60,4,10,-2,-2,-2,-2,R2,46.936275
Repyrolysis 1:1,29,10,60,4,10,-2,-2,-2,-2,R3,46.323529
Repyrolysis 1:1,30,10,360,12,150,-2,2,2,2,R1,3.129085
Repyrolysis 1:1,30,10,360,12,150,-2,2,2,2,R2,3.169935
Repyrolysis 1:1,30,10,360,12,150,-2,2,2,2,R3,3.129085
Repyrolysis 1:1,31,100,60,12,10,2,-2,2,-2,R1,56.372549
Repyrolysis 1:1,31,100,60,12,10,2,-2,2,-2,R2,58.823529
Repyrolysis 1:1,31,100,60,12,10,2,-2,2,-2,R3,61.27451
Repyrolysis 1:1,32,100,360,4,150,2,2,-2,2,R1,32.271242
Repyrolysis 1:1,32,100,360,4,150,2,2,-2,2,R2,32.51634
Repyrolysis 1:1,32,100,360,4,150,2,2,-2,2,R3,32.598039
Repyrolysis 2:1,1,50,180,8,50,0,0,0,0,R1,35.294118
Repyrolysis 2:1,1,50,180,8,50,0,0,0,0,R2,35.539216
Repyrolysis 2:1,1,50,180,8,50,0,0,0,0,R3,35.294118
Repyrolysis 2:1,2,50,180,8,50,0,0,0,0,R1,34.313725
Repyrolysis 2:1,2,50,180,8,50,0,0,0,0,R2,35.784314
Repyrolysis 2:1,2,50,180,8,50,0,0,0,0,R3,34.558824
Repyrolysis 2:1,3,50,180,8,50,0,0,0,0,R1,33.823529
Repyrolysis 2:1,3,50,180,8,50,0,0,0,0,R2,34.558824
Repyrolysis 2:1,3,50,180,8,50,0,0,0,0,R3,33.333333
Repyrolysis 2:1,4,50,180,8,50,0,0,0,0,R1,35.04902
Repyrolysis 2:1,4,50,180,8,50,0,0,0,0,R2,33.823529
Repyrolysis 2:1,4,50,180,8,50,0,0,0,0,R3,36.029412
Repyrolysis 2:1,5,25,180,8,50,-1,0,0,0,R1,23.406863
Repyrolysis 2:1,5,25,180,8,50,-1,0,0,0,R2,22.794118
Repyrolysis 2:1,5,25,180,8,50,-1,0,0,0,R3,22.54902
Repyrolysis 2:1,6,75,180,8,50,1,0,0,0,R1,40.196078
Repyrolysis 2:1,6,75,180,8,50,1,0,0,0,R2,36.27451
Repyrolysis 2:1,6,75,180,8,50,1,0,0,0,R3,37.745098
Repyrolysis 2:1,7,10,180,8,50,-2,0,0,0,R1,9.509804
Repyrolysis 2:1,7,10,180,8,50,-2,0,0,0,R2,9.387255
Repyrolysis 2:1,7,10,180,8,50,-2,0,0,0,R3,9.632353
Repyrolysis 2:1,8,100,180,8,50,2,0,0,0,R1,54.656863
Repyrolysis 2:1,8,100,180,8,50,2,0,0,0,R2,49.754902
Repyrolysis 2:1,8,100,180,8,50,2,0,0,0,R3,48.284314
Repyrolysis 2:1,9,50,120,8,50,0,-1,0,0,R1,27.696078
Repyrolysis 2:1,9,50,120,8,50,0,-1,0,0,R2,28.431373
Repyrolysis 2:1,9,50,120,8,50,0,-1,0,0,R3,29.166667
Repyrolysis 2:1,10,50,240,8,50,0,1,0,0,R1,35.784314
Repyrolysis 2:1,10,50,240,8,50,0,1,0,0,R2,36.764706
Repyrolysis 2:1,10,50,240,8,50,0,1,0,0,R3,34.558824
Repyrolysis 2:1,11,50,60,8,50,0,-2,0,0,R1,22.794118
Repyrolysis 2:1,11,50,60,8,50,0,-2,0,0,R2,21.813725
Repyrolysis 2:1,11,50,60,8,50,0,-2,0,0,R3,21.078431
Repyrolysis 2:1,12,50,360,8,50,0,2,0,0,R1,35.784314
Repyrolysis 2:1,12,50,360,8,50,0,2,0,0,R2,36.27451
Repyrolysis 2:1,12,50,360,8,50,0,2,0,0,R3,37.009804
Repyrolysis 2:1,13,50,180,6,50,0,0,-1,0,R1,40.196078
Repyrolysis 2:1,13,50,180,6,50,0,0,-1,0,R2,40.686275
Repyrolysis 2:1,13,50,180,6,50,0,0,-1,0,R3,39.460784
Repyrolysis 2:1,14,50,180,10,50,0,0,1,0,R1,32.843137
Repyrolysis 2:1,14,50,180,10,50,0,0,1,0,R2,34.558824
Repyrolysis 2:1,14,50,180,10,50,0,0,1,0,R3,35.294118
Repyrolysis 2:1,15,50,180,4,50,0,0,-2,0,R1,44.852941
Repyrolysis 2:1,15,50,180,4,50,0,0,-2,0,R2,45.098039
Repyrolysis 2:1,15,50,180,4,50,0,0,-2,0,R3,45.833333
Repyrolysis 2:1,16,50,180,12,50,0,0,2,0,R1,21.568627
Repyrolysis 2:1,16,50,180,12,50,0,0,2,0,R2,21.078431
Repyrolysis 2:1,16,50,180,12,50,0,0,2,0,R3,21.323529
Repyrolysis 2:1,17,50,180,8,20,0,0,0,-1,R1,37.377451
Repyrolysis 2:1,17,50,180,8,20,0,0,0,-1,R2,39.828431
Repyrolysis 2:1,17,50,180,8,20,0,0,0,-1,R3,41.053922
Repyrolysis 2:1,18,50,180,8,100,0,0,0,1,R1,24.264706
Repyrolysis 2:1,18,50,180,8,100,0,0,0,1,R2,23.406863
Repyrolysis 2:1,18,50,180,8,100,0,0,0,1,R3,24.019608
Repyrolysis 2:1,19,50,180,8,10,0,0,0,-2,R1,42.892157
Repyrolysis 2:1,19,50,180,8,10,0,0,0,-2,R2,42.892157
Repyrolysis 2:1,19,50,180,8,10,0,0,0,-2,R3,44.117647
Repyrolysis 2:1,20,50,180,8,150,0,0,0,2,R1,16.503268
Repyrolysis 2:1,20,50,180,8,150,0,0,0,2,R2,16.584967
Repyrolysis 2:1,20,50,180,8,150,0,0,0,2,R3,16.094771
Repyrolysis 2:1,21,25,120,6,20,-1,-1,-1,-1,R1,33.394608
Repyrolysis 2:1,21,25,120,6,20,-1,-1,-1,-1,R2,27.879902
Repyrolysis 2:1,21,25,120,6,20,-1,-1,-1,-1,R3,28.79902
Repyrolysis 2:1,22,25,240,10,100,-1,1,1,1,R1,11.029412
Repyrolysis 2:1,22,25,240,10,100,-1,1,1,1,R2,10.968137
Repyrolysis 2:1,22,25,240,10,100,-1,1,1,1,R3,11.151961
Repyrolysis 2:1,23,75,120,10,20,1,-1,1,-1,R1,56.372549
Repyrolysis 2:1,23,75,120,10,20,1,-1,1,-1,R2,42.279412
Repyrolysis 2:1,23,75,120,10,20,1,-1,1,-1,R3,53.308824
Repyrolysis 2:1,24,75,240,6,100,1,1,-1,1,R1,34.791667
Repyrolysis 2:1,24,75,240,6,100,1,1,-1,1,R2,36.25
Repyrolysis 2:1,24,75,240,6,100,1,1,-1,1,R3,35.729167
Repyrolysis 2:1,25,25,120,10,100,-1,-1,1,1,R1,10.171569
Repyrolysis 2:1,25,25,120,10,100,-1,-1,1,1,R2,10.416667
Repyrolysis 2:1,25,25,120,10,100,-1,-1,1,1,R3,9.375
Repyrolysis 2:1,26,25,240,6,20,-1,1,-1,-1,R1,53.308824
Repyrolysis 2:1,26,25,240,6,20,-1,1,-1,-1,R2,53.308824
Repyrolysis 2:1,26,25,240,6,20,-1,1,-1,-1,R3,56.678922
Repyrolysis 2:1,27,75,120,6,100,1,-1,-1,1,R1,32.107843
Repyrolysis 2:1,27,75,120,6,100,1,-1,-1,1,R2,34.926471
Repyrolysis 2:1,27,75,120,6,100,1,-1,-1,1,R3,34.068627
Repyrolysis 2:1,28,75,240,10,20,1,1,1,-1,R1,36.151961
Repyrolysis 2:1,28,75,240,10,20,1,1,1,-1,R2,41.053922
Repyrolysis 2:1,28,75,240,10,20,1,1,1,-1,R3,39.215686
Repyrolysis 2:1,29,10,60,4,10,-2,-2,-2,-2,R1,46.323529
Repyrolysis 2:1,29,10,60,4,10,-2,-2,-2,-2,R2,46.936275
Repyrolysis 2:1,29,10,60,4,10,-2,-2,-2,-2,R3,46.323529
Repyrolysis 2:1,30,10,360,12,150,-2,2,2,2,R1,3.129085
Repyrolysis 2:1,30,10,360,12,150,-2,2,2,2,R2,3.169935
Repyrolysis 2:1,30,10,360,12,150,-2,2,2,2,R3,3.129085
Repyrolysis 2:1,31,100,60,12,10,2,-2,2,-2,R1,56.372549
Repyrolysis 2:1,31,100,60,12,10,2,-2,2,-2,R2,49.2
Repyrolysis 2:1,31,100,60,12,10,2,-2,2,-2,R3,49.019608
Repyrolysis 2:1,32,100,360,4,150,2,2,-2,2,R1,32.271242
Repyrolysis 2:1,32,100,360,4,150,2,2,-2,2,R2,32.51634
Repyrolysis 2:1,32,100,360,4,150,2,2,-2,2,R3,32.598039
BC:MgCO3 1:1,1,50,180,8,50,0,0,0,0,R1,41.421569
BC:MgCO3 1:1,1,50,180,8,50,0,0,0,0,R2,41.176471
BC:MgCO3 1:1,1,50,180,8,50,0,0,0,0,R3,40.441176
BC:MgCO3 1:1,2,50,180,8,50,0,0,0,0,R1,39.460784
BC:MgCO3 1:1,2,50,180,8,50,0,0,0,0,R2,41.176471
BC:MgCO3 1:1,2,50,180,8,50,0,0,0,0,R3,40.686275
BC:MgCO3 1:1,3,50,180,8,50,0,0,0,0,R1,41.666667
BC:MgCO3 1:1,3,50,180,8,50,0,0,0,0,R2,41.666667
BC:MgCO3 1:1,3,50,180,8,50,0,0,0,0,R3,41.421569
BC:MgCO3 1:1,4,50,180,8,50,0,0,0,0,R1,41.421569
BC:MgCO3 1:1,4,50,180,8,50,0,0,0,0,R2,41.911765
BC:MgCO3 1:1,4,50,180,8,50,0,0,0,0,R3,42.401961
BC:MgCO3 1:1,5,25,180,8,50,-1,0,0,0,R1,21.078431
BC:MgCO3 1:1,5,25,180,8,50,-1,0,0,0,R2,20.710784
BC:MgCO3 1:1,5,25,180,8,50,-1,0,0,0,R3,20.833333
BC:MgCO3 1:1,6,75,180,8,50,1,0,0,0,R1,48.529412
BC:MgCO3 1:1,6,75,180,8,50,1,0,0,0,R2,47.303922
BC:MgCO3 1:1,6,75,180,8,50,1,0,0,0,R3,40.441176
BC:MgCO3 1:1,7,10,180,8,50,-2,0,0,0,R1,8.529412
BC:MgCO3 1:1,7,10,180,8,50,-2,0,0,0,R2,8.77451
BC:MgCO3 1:1,7,10,180,8,50,-2,0,0,0,R3,8.77451
BC:MgCO3 1:1,8,100,180,8,50,2,0,0,0,R1,65.441176
BC:MgCO3 1:1,8,100,180,8,50,2,0,0,0,R2,66.176471
BC:MgCO3 1:1,8,100,180,8,50,2,0,0,0,R3,64.95098
BC:MgCO3 1:1,9,50,120,8,50,0,-1,0,0,R1,40.196078
BC:MgCO3 1:1,9,50,120,8,50,0,-1,0,0,R2,39.705882
BC:MgCO3 1:1,9,50,120,8,50,0,-1,0,0,R3,40.441176
BC:MgCO3 1:1,10,50,240,8,50,0,1,0,0,R1,42.401961
BC:MgCO3 1:1,10,50,240,8,50,0,1,0,0,R2,41.911765
BC:MgCO3 1:1,10,50,240,8,50,0,1,0,0,R3,43.137255
BC:MgCO3 1:1,11,50,60,8,50,0,-2,0,0,R1,35.784314
BC:MgCO3 1:1,11,50,60,8,50,0,-2,0,0,R2,34.558824
BC:MgCO3 1:1,11,50,60,8,50,0,-2,0,0,R3,32.843137
BC:MgCO3 1:1,12,50,360,8,50,0,2,0,0,R1,44.852941
BC:MgCO3 1:1,12,50,360,8,50,0,2,0,0,R2,43.382353
BC:MgCO3 1:1,12,50,360,8,50,0,2,0,0,R3,46.078431
BC:MgCO3 1:1,13,50,180,6,50,0,0,-1,0,R1,44.852941
BC:MgCO3 1:1,13,50,180,6,50,0,0,-1,0,R2,44.117647
BC:MgCO3 1:1,13,50,180,6,50,0,0,-1,0,R3,45.343137
BC:MgCO3 1:1,14,50,180,10,50,0,0,1,0,R1,34.313725
BC:MgCO3 1:1,14,50,180,10,50,0,0,1,0,R2,34.803922
BC:MgCO3 1:1,14,50,180,10,50,0,0,1,0,R3,34.068627
BC:MgCO3 1:1,15,50,180,4,50,0,0,-2,0,R1,46.323529
BC:MgCO3 1:1,15,50,180,4,50,0,0,-2,0,R2,45.588235
BC:MgCO3 1:1,15,50,180,4,50,0,0,-2,0,R3,44.607843
BC:MgCO3 1:1,16,50,180,12,50,0,0,2,0,R1,28.921569
BC:MgCO3 1:1,16,50,180,12,50,0,0,2,0,R2,29.411765
BC:MgCO3 1:1,16,50,180,12,50,0,0,2,0,R3,27.696078
BC:MgCO3 1:1,17,50,180,8,20,0,0,0,-1,R1,77.818627
BC:MgCO3 1:1,17,50,180,8,20,0,0,0,-1,R2,74.142157
BC:MgCO3 1:1,17,50,180,8,20,0,0,0,-1,R3,82.107843
BC:MgCO3 1:1,18,50,180,8,100,0,0,0,1,R1,22.916667
BC:MgCO3 1:1,18,50,180,8,100,0,0,0,1,R2,23.039216
BC:MgCO3 1:1,18,50,180,8,100,0,0,0,1,R3,23.039216
BC:MgCO3 1:1,19,50,180,8,10,0,0,0,-2,R1,98.039216
BC:MgCO3 1:1,19,50,180,8,10,0,0,0,-2,R2,91.911765
BC:MgCO3 1:1,19,50,180,8,10,0,0,0,-2,R3,84.558824
BC:MgCO3 1:1,20,50,180,8,150,0,0,0,2,R1,15.849673
BC:MgCO3 1:1,20,50,180,8,150,0,0,0,2,R2,15.767974
BC:MgCO3 1:1,20,50,180,8,150,0,0,0,2,R3,16.094771
BC:MgCO3 1:1,21,25,120,6,20,-1,-1,-1,-1,R1,49.632353
BC:MgCO3 1:1,21,25,120,6,20,-1,-1,-1,-1,R2,50.857843
BC:MgCO3 1:1,21,25,120,6,20,-1,-1,-1,-1,R3,49.938725
BC:MgCO3 1:1,22,25,240,10,100,-1,1,1,1,R1,11.764706
BC:MgCO3 1:1,22,25,240,10,100,-1,1,1,1,R2,12.071078
BC:MgCO3 1:1,22,25,240,10,100,-1,1,1,1,R3,11.887255
BC:MgCO3 1:1,23,75,120,10,20,1,-1,1,-1,R1,64.338235
BC:MgCO3 1:1,23,75,120,10,20,1,-1,1,-1,R2,62.5
BC:MgCO3 1:1,23,75,120,10,20,1,-1,1,-1,R3,65.563725
BC:MgCO3 1:1,24,75,240,6,100,1,1,-1,1,R1,35.539216
BC:MgCO3 1:1,24,75,240,6,100,1,1,-1,1,R2,34.681373
BC:MgCO3 1:1,24,75,240,6,100,1,1,-1,1,R3,35.416667
BC:MgCO3 1:1,25,25,120,10,100,-1,-1,1,1,R1,11.090686
BC:MgCO3 1:1,25,25,120,10,100,-1,-1,1,1,R2,11.27451
BC:MgCO3 1:1,25,25,120,10,100,-1,-1,1,1,R3,11.397059
BC:MgCO3 1:1,26,25,240,6,20,-1,1,-1,-1,R1,57.291667
BC:MgCO3 1:1,26,25,240,6,20,-1,1,-1,-1,R2,50.857843
BC:MgCO3 1:1,26,25,240,6,20,-1,1,-1,-1,R3,51.164216
BC:MgCO3 1:1,27,75,120,6,100,1,-1,-1,1,R1,35.04902
BC:MgCO3 1:1,27,75,120,6,100,1,-1,-1,1,R2,35.661765
BC:MgCO3 1:1,27,75,120,6,100,1,-1,-1,1,R3,36.029412
BC:MgCO3 1:1,28,75,240,10,20,1,1,1,-1,R1,82.107843
BC:MgCO3 1:1,28,75,240,10,20,1,1,1,-1,R2,83.333333
BC:MgCO3 1:1,28,75,240,10,20,1,1,1,-1,R3,85.784314
BC:MgCO3 1:1,29,10,60,4,10,-2,-2,-2,-2,R1,24.264706
BC:MgCO3 1:1,29,10,60,4,10,-2,-2,-2,-2,R2,25.490196
BC:MgCO3 1:1,29,10,60,4,10,-2,-2,-2,-2,R3,23.039216
BC:MgCO3 1:1,30,10,360,12,150,-2,2,2,2,R1,2.924837
BC:MgCO3 1:1,30,10,360,12,150,-2,2,2,2,R2,2.965686
BC:MgCO3 1:1,30,10,360,12,150,-2,2,2,2,R3,2.965686
BC:MgCO3 1:1,31,100,60,12,10,2,-2,2,-2,R1,66.176471
BC:MgCO3 1:1,31,100,60,12,10,2,-2,2,-2,R2,66.176471
BC:MgCO3 1:1,31,100,60,12,10,2,-2,2,-2,R3,77.205882
BC:MgCO3 1:1,32,100,360,4,150,2,2,-2,2,R1,31.617647
BC:MgCO3 1:1,32,100,360,4,150,2,2,-2,2,R2,32.434641
BC:MgCO3 1:1,32,100,360,4,150,2,2,-2,2,R3,32.189542
BC:MgCO3 2:1,1,50,180,8,50,0,0,0,0,R1,19.117647
BC:MgCO3 2:1,1,50,180,8,50,0,0,0,0,R2,17.892157
BC:MgCO3 2:1,1,50,180,8,50,0,0,0,0,R3,19.607843
BC:MgCO3 2:1,2,50,180,8,50,0,0,0,0,R1,21.813725
BC:MgCO3 2:1,2,50,180,8,50,0,0,0,0,R2,22.058824
BC:MgCO3 2:1,2,50,180,8,50,0,0,0,0,R3,20.343137
BC:MgCO3 2:1,3,50,180,8,50,0,0,0,0,R1,19.117647
BC:MgCO3 2:1,3,50,180,8,50,0,0,0,0,R2,18.872549
BC:MgCO3 2:1,3,50,180,8,50,0,0,0,0,R3,17.647059
BC:MgCO3 2:1,4,50,180,8,50,0,0,0,0,R1,22.303922
BC:MgCO3 2:1,4,50,180,8,50,0,0,0,0,R2,21.813725
BC:MgCO3 2:1,4,50,180,8,50,0,0,0,0,R3,21.813725
BC:MgCO3 2:1,5,25,180,8,50,-1,0,0,0,R1,9.558824
BC:MgCO3 2:1,5,25,180,8,50,-1,0,0,0,R2,8.946078
BC:MgCO3 2:1,5,25,180,8,50,-1,0,0,0,R3,9.803922
BC:MgCO3 2:1,6,75,180,8,50,1,0,0,0,R1,27.45098
BC:MgCO3 2:1,6,75,180,8,50,1,0,0,0,R2,31.862745
BC:MgCO3 2:1,6,75,180,8,50,1,0,0,0,R3,35.294118
BC:MgCO3 2:1,7,10,180,8,50,-2,0,0,0,R1,6.20098
BC:MgCO3 2:1,7,10,180,8,50,-2,0,0,0,R2,4.362745
BC:MgCO3 2:1,7,10,180,8,50,-2,0,0,0,R3,5.955882
BC:MgCO3 2:1,8,100,180,8,50,2,0,0,0,R1,26.960784
BC:MgCO3 2:1,8,100,180,8,50,2,0,0,0,R2,32.598039
BC:MgCO3 2:1,8,100,180,8,50,2,0,0,0,R3,31.372549
BC:MgCO3 2:1,9,50,120,8,50,0,-1,0,0,R1,18.137255
BC:MgCO3 2:1,9,50,120,8,50,0,-1,0,0,R2,19.362745
BC:MgCO3 2:1,9,50,120,8,50,0,-1,0,0,R3,20.098039
BC:MgCO3 2:1,10,50,240,8,50,0,1,0,0,R1,28.186275
BC:MgCO3 2:1,10,50,240,8,50,0,1,0,0,R2,31.127451
BC:MgCO3 2:1,10,50,240,8,50,0,1,0,0,R3,28.676471
BC:MgCO3 2:1,11,50,60,8,50,0,-2,0,0,R1,14.215686
BC:MgCO3 2:1,11,50,60,8,50,0,-2,0,0,R2,15.686275
BC:MgCO3 2:1,11,50,60,8,50,0,-2,0,0,R3,13.970588
BC:MgCO3 2:1,12,50,360,8,50,0,2,0,0,R1,32.843137
BC:MgCO3 2:1,12,50,360,8,50,0,2,0,0,R2,32.843137
BC:MgCO3 2:1,12,50,360,8,50,0,2,0,0,R3,33.333333
BC:MgCO3 2:1,13,50,180,6,50,0,0,-1,0,R1,15.196078
BC:MgCO3 2:1,13,50,180,6,50,0,0,-1,0,R2,15.686275
BC:MgCO3 2:1,13,50,180,6,50,0,0,-1,0,R3,14.705882
BC:MgCO3 2:1,14,50,180,10,50,0,0,1,0,R1,19.117647
BC:MgCO3 2:1,14,50,180,10,50,0,0,1,0,R2,18.627451
BC:MgCO3 2:1,14,50,180,10,50,0,0,1,0,R3,17.892157
BC:MgCO3 2:1,15,50,180,4,50,0,0,-2,0,R1,29.411765
BC:MgCO3 2:1,15,50,180,4,50,0,0,-2,0,R2,28.676471
BC:MgCO3 2:1,15,50,180,4,50,0,0,-2,0,R3,27.696078
BC:MgCO3 2:1,16,50,180,12,50,0,0,2,0,R1,15.441176
BC:MgCO3 2:1,16,50,180,12,50,0,0,2,0,R2,18.627451
BC:MgCO3 2:1,16,50,180,12,50,0,0,2,0,R3,18.872549
BC:MgCO3 2:1,17,50,180,8,20,0,0,0,-1,R1,27.573529
BC:MgCO3 2:1,17,50,180,8,20,0,0,0,-1,R2,19.607843
BC:MgCO3 2:1,17,50,180,8,20,0,0,0,-1,R3,25.735294
BC:MgCO3 2:1,18,50,180,8,100,0,0,0,1,R1,21.997549
BC:MgCO3 2:1,18,50,180,8,100,0,0,0,1,R2,22.610294
BC:MgCO3 2:1,18,50,180,8,100,0,0,0,1,R3,22.242647
BC:MgCO3 2:1,19,50,180,8,10,0,0,0,-2,R1,35.539216
BC:MgCO3 2:1,19,50,180,8,10,0,0,0,-2,R2,41.666667
BC:MgCO3 2:1,19,50,180,8,10,0,0,0,-2,R3,36.764706
BC:MgCO3 2:1,20,50,180,8,150,0,0,0,2,R1,16.339869
BC:MgCO3 2:1,20,50,180,8,150,0,0,0,2,R2,16.013072
BC:MgCO3 2:1,20,50,180,8,150,0,0,0,2,R3,16.094771
BC:MgCO3 2:1,21,25,120,6,20,-1,-1,-1,-1,R1,0.919118
BC:MgCO3 2:1,21,25,120,6,20,-1,-1,-1,-1,R2,3.676471
BC:MgCO3 2:1,21,25,120,6,20,-1,-1,-1,-1,R3,3.063725
BC:MgCO3 2:1,22,25,240,10,100,-1,1,1,1,R1,12.132353
BC:MgCO3 2:1,22,25,240,10,100,-1,1,1,1,R2,12.071078
BC:MgCO3 2:1,22,25,240,10,100,-1,1,1,1,R3,11.887255
BC:MgCO3 2:1,23,75,120,10,20,1,-1,1,-1,R1,14.705882
BC:MgCO3 2:1,23,75,120,10,20,1,-1,1,-1,R2,13.480392
BC:MgCO3 2:1,23,75,120,10,20,1,-1,1,-1,R3,19.607843
BC:MgCO3 2:1,24,75,240,6,100,1,1,-1,1,R1,30.514706
BC:MgCO3 2:1,24,75,240,6,100,1,1,-1,1,R2,31.004902
BC:MgCO3 2:1,24,75,240,6,100,1,1,-1,1,R3,29.901961
BC:MgCO3 2:1,25,25,120,10,100,-1,-1,1,1,R1,11.887255
BC:MgCO3 2:1,25,25,120,10,100,-1,-1,1,1,R2,12.316176
BC:MgCO3 2:1,25,25,120,10,100,-1,-1,1,1,R3,12.254902
BC:MgCO3 2:1,26,25,240,6,20,-1,1,-1,-1,R1,9.497549
BC:MgCO3 2:1,26,25,240,6,20,-1,1,-1,-1,R2,9.803922
BC:MgCO3 2:1,26,25,240,6,20,-1,1,-1,-1,R3,4.901961
BC:MgCO3 2:1,27,75,120,6,100,1,-1,-1,1,R1,32.352941
BC:MgCO3 2:1,27,75,120,6,100,1,-1,-1,1,R2,34.681373
BC:MgCO3 2:1,27,75,120,6,100,1,-1,-1,1,R3,33.333333
BC:MgCO3 2:1,28,75,240,10,20,1,1,1,-1,R1,22.058824
BC:MgCO3 2:1,28,75,240,10,20,1,1,1,-1,R2,43.504902
BC:MgCO3 2:1,28,75,240,10,20,1,1,1,-1,R3,26.960784
BC:MgCO3 2:1,29,10,60,4,10,-2,-2,-2,-2,R1,1.593137
BC:MgCO3 2:1,29,10,60,4,10,-2,-2,-2,-2,R2,2.205882
BC:MgCO3 2:1,29,10,60,4,10,-2,-2,-2,-2,R3,2.818627
BC:MgCO3 2:1,30,10,360,12,150,-2,2,2,2,R1,3.129085
BC:MgCO3 2:1,30,10,360,12,150,-2,2,2,2,R2,3.129085
BC:MgCO3 2:1,30,10,360,12,150,-2,2,2,2,R3,3.088235
BC:MgCO3 2:1,31,100,60,12,10,2,-2,2,-2,R1,55.147059
BC:MgCO3 2:1,31,100,60,12,10,2,-2,2,-2,R2,49.019608
BC:MgCO3 2:1,31,100,60,12,10,2,-2,2,-2,R3,49.019608
BC:MgCO3 2:1,32,100,360,4,150,2,2,-2,2,R1,31.045752
BC:MgCO3 2:1,32,100,360,4,150,2,2,-2,2,R2,31.699346
BC:MgCO3 2:1,32,100,360,4,150,2,2,-2,2,R3,31.20915"""
d = pd.read_csv(io.StringIO(CSV))
MATS = list(d.mat.unique())
X4 = ['x1', 'x2', 'x3', 'x4']
NAME = {'x1': 'A:Conc', 'x2': 'B:Time', 'x3': 'C:pH', 'x4': 'D:Dose'}
LEVELS = {'x1': ('Concentration (ppm)', [10, 25, 50, 75, 100]), 'x2': ('Time (min)', [60, 120, 180, 240, 360]),
          'x3': ('pH', [4, 6, 8, 10, 12]), 'x4': ('Dosage (mg)', [10, 20, 50, 100, 150])}
to_actual = lambda k, c: np.interp(c, [-2, -1, 0, 1, 2], LEVELS[k][1])
to_coded = lambda k, v: np.interp(v, LEVELS[k][1], [-2, -1, 0, 1, 2])
d['cond'] = np.where(d.run <= 4, 0, d.run)   # centre runs 1-4 = one condition
Y_ORIG = d.y.copy()                          # checked at the end: measured values must stay unchanged
print(d.shape, d.groupby('mat').size().to_dict())

## Model definitions (same as v1/v2)

In [ ]:
LIN = X4
SQ = [f'{a}^2' for a in X4]
INT = [f'{a}*{b}' for i, a in enumerate(X4) for b in X4[i + 1:]]
ALL = LIN + SQ + INT

def design(df, terms):
    X = pd.DataFrame(index=df.index)
    for t in terms:
        if t.endswith('^2'): X[t] = df[t[:-2]] ** 2
        elif '*' in t: a, b = t.split('*'); X[t] = df[a] * df[b]
        else: X[t] = df[t]
    return sm.add_constant(X, has_constant='add')

def terms_of(m): return m.model.exog_names[1:]

def fit_rsm(df, reduce=True, w=None):
    """Quadratic RSM. reduce=True: backward elimination (p > 0.05), keeping hierarchy. w: optional weights (WLS)."""
    ols = lambda X: sm.OLS(df.y, X).fit() if w is None else sm.WLS(df.y, X, weights=w).fit()
    terms = ALL.copy()
    while reduce:
        m = ols(design(df, terms))
        needed = {x for t in terms if t not in LIN for x in X4 if x in t}
        p = m.pvalues[[t for t in terms if t not in needed]]
        if p.empty or p.max() <= 0.05: break
        terms.remove(p.idxmax())
    return ols(design(df, terms))

def rsm_predict(m, df): return m.predict(design(df, terms_of(m))).values

def ann_X(df, cols=X4):
    onehot = pd.get_dummies(df.mat).reindex(columns=MATS, fill_value=0).astype(float)
    return pd.concat([df[cols].astype(float), onehot], axis=1).values

def make_ann(h, a, seed):
    net = MLPRegressor((h,), activation='tanh', solver='lbfgs', alpha=a, max_iter=5000, random_state=seed)
    return TransformedTargetRegressor(make_pipeline(StandardScaler(), net), transformer=StandardScaler())

GRID = [(h, a) for h in (4, 8, 12) for a in (1e-3, 1e-2, 1e-1)]

def fit_ann(df, n_ens=5, cols=X4, target=None, grid=GRID, k=4):
    """One ANN for all materials (material = one-hot input). Neurons and L2 tuned by grouped
    inner CV on THIS data only; the final model is an ensemble of n_ens seeds."""
    X, g = ann_X(df, cols), df.cond.values
    y = df.y.values if target is None else np.asarray(target)
    folds = list(GroupKFold(k).split(X, y, g))
    def cv_mse(h, a):
        return np.mean([mean_squared_error(y[va], make_ann(h, a, SEED).fit(X[tr], y[tr]).predict(X[va])) for tr, va in folds])
    h, a = min(grid, key=lambda p: cv_mse(*p))
    return {'nets': [make_ann(h, a, SEED + i).fit(X, y) for i in range(n_ens)], 'h': h, 'alpha': a, 'cols': cols}

def ann_predict(model, df): return np.mean([n.predict(ann_X(df, model['cols'])) for n in model['nets']], axis=0)

def metrics(a, p): return {'R2': r2_score(a, p), 'RMSE': np.sqrt(mean_squared_error(a, p)), 'MAE': mean_absolute_error(a, p)}

MODELS = ['RSM full', 'RSM reduced', 'ANN']

def score(pred_rows, models=MODELS):
    """Metrics on run means (actual = mean of the 3 replicates), per material and overall."""
    g = pred_rows.groupby(['mat', 'run'])[['y'] + models].mean().reset_index()
    out = [{'Material': m, 'Model': k, **metrics(s.y, s[k])} for m, s in g.groupby('mat', sort=False) for k in models]
    out += [{'Material': 'ALL', 'Model': k, **metrics(g.y, g[k])} for k in models]
    return pd.DataFrame(out).round(3)

## Hold-out split (same as v1/v2: test runs 5, 12, 15, 18, 24, 27)

In [ ]:
rng = np.random.default_rng(SEED)
# Stratified random draw: 1 run from each factor's axial block + 2 from the factorial block, so training
# keeps information on every factor. Centre (1-4) and extreme corners (29-32) always stay in training.
BLOCKS = [(range(5, 9), 1), (range(9, 13), 1), (range(13, 17), 1), (range(17, 21), 1), (range(21, 29), 2)]
while True:
    test_conds = sorted(int(c) for b, k in BLOCKS for c in rng.choice(list(b), k, replace=False))
    tr0 = d[~d.cond.isin(test_conds) & (d.mat == MATS[0])]
    if np.linalg.matrix_rank(design(tr0, ALL).values) == len(ALL) + 1: break   # full quadratic still estimable
d['set'] = np.where(d.cond.isin(test_conds), 'test', 'train')
train, test = d[d.set == 'train'], d[d.set == 'test']
assert not set(train.cond) & set(test.cond), 'leakage: a condition is in both sets'
print('Test runs (unseen):', test_conds)
print('Train rows:', len(train), '| Test rows:', len(test))
print(d[d.set == 'test'].drop_duplicates('run')[['run', 'conc', 'time', 'pH', 'dose']].to_string(index=False))

## Candidate models from v2

In [ ]:
d['logR'] = np.log(d.conc / d.dose)   # input derived from existing columns
train, test = d[d.set == 'train'], d[d.set == 'test']
PHYS = X4 + ['logR']
to_T = lambda df, q: np.asarray(q) * df.dose.values / df.conc.values     # qe -> removal-proportional target
from_T = lambda df, t: np.asarray(t) * df.conc.values / df.dose.values   # exact inverse
GRID2 = [(h, a) for h in (4, 8, 12) for a in (1e-2, 1e-1)]
ENRICHED = MATS[1:]

def by_mat(tr, te, f, mats=MATS):
    out = pd.Series(np.nan, index=te.index)
    for m in mats:
        a, b = tr[tr.mat == m], te[te.mat == m]
        if len(b): out[b.index] = f(a, b)
    return out.values

def rsm_weighted(a, b):
    v = a.groupby('cond').y.var()
    v = v.clip(lower=v.median())          # floor so no condition gets extreme weight
    return rsm_predict(fit_rsm(a, w=1 / a.cond.map(v).values), b)

def rsm_removal(a, b): return from_T(b, rsm_predict(fit_rsm(a.assign(y=to_T(a, a.y))), b))

def gpr(a, b, removal=False):
    kern = ConstantKernel(1.0) * RBF(np.ones(len(PHYS)), (1e-2, 1e2)) + WhiteKernel(0.1, (1e-5, 10))
    sc = StandardScaler().fit(a[PHYS])
    y = to_T(a, a.y) if removal else a.y.values
    g = GaussianProcessRegressor(kern, normalize_y=True, n_restarts_optimizer=2, random_state=SEED).fit(sc.transform(a[PHYS]), y)
    p = g.predict(sc.transform(b[PHYS]))
    return from_T(b, p) if removal else p

def hgb(tr, te, mono):
    X = lambda df: np.column_stack([df[PHYS].values, df.mat.map({m: i for i, m in enumerate(MATS)}).values])
    Xa, ya = X(tr), tr.y.values
    cst = [1, 0, 0, -1, 1, 0] if mono else None   # conc up, dose down, conc/dose up
    mk = lambda lr, leaves: HistGradientBoostingRegressor(learning_rate=lr, max_leaf_nodes=leaves, max_iter=300, min_samples_leaf=5,
                                                          monotonic_cst=cst, categorical_features=[5], random_state=SEED)
    folds = list(GroupKFold(4).split(Xa, ya, tr.cond.values))
    cfgs = [(lr, leaves) for lr in (0.05, 0.1) for leaves in (7, 15)]
    best = min(cfgs, key=lambda c: np.mean([mean_squared_error(ya[v], mk(*c).fit(Xa[t], ya[t]).predict(Xa[v])) for t, v in folds]))
    return mk(*best).fit(Xa, ya).predict(X(te))

def mono_boost(tr, te):
    out = pd.Series(np.nan, index=te.index)
    e_tr, e_te = tr[tr.mat.isin(ENRICHED)], te[te.mat.isin(ENRICHED)]
    out[e_te.index] = hgb(e_tr, e_te, True)
    b_tr, b_te = tr[~tr.mat.isin(ENRICHED)], te[~te.mat.isin(ENRICHED)]
    if len(b_te): out[b_te.index] = hgb(b_tr, b_te, False)   # control: no constraint
    return out.values

def hybrid(tr, te):
    base = lambda a, b: rsm_predict(fit_rsm(a), b)
    resid = tr.y.values - by_mat(tr, tr, base)
    return by_mat(tr, te, base) + ann_predict(fit_ann(tr, 10, PHYS, target=resid, grid=GRID2, k=3), te)

CANDIDATES = {
    'RSM weighted': lambda tr, te: by_mat(tr, te, rsm_weighted),
    'RSM removal-space': lambda tr, te: by_mat(tr, te, rsm_removal),
    'ANN + conc/dose': lambda tr, te: ann_predict(fit_ann(tr, 10, PHYS, grid=GRID2, k=3), te),
    'ANN removal-space': lambda tr, te: from_T(te, ann_predict(fit_ann(tr, 10, PHYS, target=to_T(tr, tr.y), grid=GRID2, k=3), te)),
    'GPR + conc/dose': lambda tr, te: by_mat(tr, te, gpr),
    'GPR removal-space': lambda tr, te: by_mat(tr, te, lambda a, b: gpr(a, b, True)),
    'Monotonic boosting': mono_boost,
    'RSM + ANN residual': hybrid,
}
NEW = list(CANDIDATES) + ['Ensemble (ANN+GPR)']
ALLM = MODELS + NEW

def run_candidates(tr, te):
    warnings.filterwarnings('ignore')
    out = te.copy()
    for k, f in CANDIDATES.items(): out[k] = f(tr, te)
    out['Ensemble (ANN+GPR)'] = out[['ANN + conc/dose', 'GPR + conc/dose']].mean(axis=1)
    return out

## New in v3: ANN variants and a convergence check
- **2-layer ANN:** two hidden layers of equal size, L-BFGS solver, iteration limit raised to 20 000.
- **Adam + early stopping:** trained with the Adam optimiser. 15% of the *training* rows are set aside internally, and training stops once their error has not improved for 50 epochs.

In [ ]:
def make_ann_v3(h, a, seed, layers=1, solver='lbfgs'):
    kw = (dict(early_stopping=True, validation_fraction=0.15, n_iter_no_change=50, max_iter=5000)
          if solver == 'adam' else dict(max_iter=20000))
    net = MLPRegressor((h,) * layers, activation='tanh', solver=solver, alpha=a, random_state=seed, **kw)
    return TransformedTargetRegressor(make_pipeline(StandardScaler(), net), transformer=StandardScaler())

def fit_ann_v3(df, layers, solver, n_ens=10, cols=PHYS, grid=GRID2, k=3):
    """Same tuning rules as v1/v2: grid chosen by grouped CV on this data only, then a seed ensemble."""
    X, y, g = ann_X(df, cols), df.y.values, df.cond.values
    folds = list(GroupKFold(k).split(X, y, g))
    cv = lambda h, a: np.mean([mean_squared_error(y[v], make_ann_v3(h, a, SEED, layers, solver).fit(X[t], y[t]).predict(X[v])) for t, v in folds])
    h, a = min(grid, key=lambda p: cv(*p))
    return {'nets': [make_ann_v3(h, a, SEED + i, layers, solver).fit(X, y) for i in range(n_ens)], 'cols': cols}

CANDIDATES = {
    'RSM full': lambda tr, te: by_mat(tr, te, lambda a, b: rsm_predict(fit_rsm(a, False), b)),
    'RSM reduced': lambda tr, te: by_mat(tr, te, lambda a, b: rsm_predict(fit_rsm(a), b)),
    'ANN': lambda tr, te: ann_predict(fit_ann(tr), te),
    **CANDIDATES,
    'ANN 2-layer + conc/dose': lambda tr, te: ann_predict(fit_ann_v3(tr, 2, 'lbfgs', grid=[(h, a) for h in (4, 8) for a in (1e-2, 1e-1)]), te),
    'ANN Adam early-stop + conc/dose': lambda tr, te: ann_predict(fit_ann_v3(tr, 1, 'adam'), te),
}
ENS = 'Ensemble (ANN+GPR)'
ALLM = list(CANDIDATES) + [ENS]

def run_candidates(tr, te, names=None):
    warnings.filterwarnings('ignore')
    names = ALLM if names is None else names
    need = set(names) | ({'ANN + conc/dose', 'GPR + conc/dose'} if ENS in names else set())
    out = te.copy()
    for k in CANDIDATES:
        if k in need: out[k] = CANDIDATES[k](tr, te)
    if ENS in names: out[ENS] = out[['ANN + conc/dose', 'GPR + conc/dose']].mean(axis=1)
    return out

# Convergence check: iterations actually used by the v1 ANN settings (L-BFGS, limit 5000), trained on all data
rows = []
X_all, y_all = ann_X(d, X4), d.y.values
for h, a in GRID:
    for s in range(3):
        n = make_ann(h, a, SEED + s).fit(X_all, y_all).regressor_.named_steps['mlpregressor']
        rows.append({'Neurons': h, 'L2 alpha': a, 'Seed': SEED + s, 'Iterations used': n.n_iter_, 'Limit': n.max_iter,
                     'Hit limit?': 'YES' if n.n_iter_ >= n.max_iter else 'no'})
convergence = pd.DataFrame(rows)
print('Networks that hit the iteration limit:', (convergence['Hit limit?'] == 'YES').sum(), 'of', len(convergence))
convergence

## Repeated stratified grouped 5-fold CV (selects the best model)
Strata = design block of each run. Groups = run condition (centre runs 1–4 count as one condition). Every candidate is fitted and tuned from scratch inside each training fold.

In [ ]:
BLOCK = lambda r: ('centre' if r <= 4 else 'conc' if r <= 8 else 'time' if r <= 12 else 'pH' if r <= 16
                   else 'dose' if r <= 20 else 'factorial' if r <= 28 else 'corner')
d['block'] = d.run.map(BLOCK)
REPEATS = 3
FOLDS = []
for rep in range(REPEATS):
    for k, (tr, te) in enumerate(StratifiedGroupKFold(5, shuffle=True, random_state=SEED + rep).split(d, d.block, d.cond)):
        assert not set(d.cond.iloc[tr]) & set(d.cond.iloc[te]), 'leakage'
        FOLDS.append((rep, tr, te))
print(len(FOLDS), 'folds; test conditions per fold:', [d.cond.iloc[te].nunique() for _, _, te in FOLDS[:5]])

res = Parallel(n_jobs=-1, verbose=5)(delayed(run_candidates)(d.iloc[tr], d.iloc[te]) for _, tr, te in FOLDS)
cv_pred = pd.concat([r.assign(rep=rep) for (rep, _, _), r in zip(FOLDS, res)])

# Score each repeat on its out-of-fold predictions (every run predicted once per repeat), then average over repeats
per_rep = pd.concat([score(g, ALLM).assign(rep=rep) for rep, g in cv_pred.groupby('rep')])
cv_all = per_rep[per_rep.Material == 'ALL'].groupby('Model')[['R2', 'RMSE', 'MAE']].agg(['mean', 'std'])
cv_all.columns = [f'5-fold {a} {b}' for a, b in cv_all.columns]
cv_all = cv_all.sort_values('5-fold RMSE mean')
BEST = cv_all.index[0]
print('Best model by repeated stratified 5-fold CV (lowest RMSE):', BEST)
cv_all.round(3)

## Confirm the chosen model on the hold-out test and leave-one-condition-out

In [ ]:
hold = run_candidates(train, test)
t_all = score(hold, ALLM)
CONFIRM = list(dict.fromkeys([BEST, 'RSM reduced', 'ANN']))
res = Parallel(n_jobs=-1, verbose=5)(delayed(run_candidates)(d[d.cond != c], d[d.cond == c], CONFIRM) for c in sorted(d.cond.unique()))
loco = pd.concat(res)
l_conf = score(loco, CONFIRM)
assert d.y.equals(Y_ORIG), 'measured values changed!'
print('Check passed: all 480 measured qe values are unchanged.')

summary = (cv_all.join(t_all[t_all.Material == 'ALL'].set_index('Model')[['R2', 'RMSE', 'MAE']].add_prefix('Hold-out '))
           .join(l_conf[l_conf.Material == 'ALL'].set_index('Model')[['R2', 'RMSE', 'MAE']].add_prefix('LOCO ')))
display(summary.round(3))
per_material = (t_all[t_all.Model.isin(CONFIRM)].pivot(index='Material', columns='Model', values='R2').add_prefix('Hold-out R2 ')
                .join(l_conf.pivot(index='Material', columns='Model', values='R2').add_prefix('LOCO R2 ')))
per_material.round(3)

## Actual vs predicted with the best model (each run predicted while unseen, LOCO)

In [ ]:
avp = (d.groupby(['mat', 'run']).agg(conc=('conc', 'first'), time=('time', 'first'), pH=('pH', 'first'), dose=('dose', 'first'),
                                     actual=('y', 'mean'), actual_sd=('y', 'std'))
       .join(loco.groupby(['mat', 'run'])[CONFIRM].mean().add_suffix(' unseen (LOCO)')).reset_index().round(3))
fig, axs = plt.subplots(1, 5, figsize=(24, 5))
for ax, m in zip(axs, MATS):
    s = avp[avp.mat == m]
    for col, mk in [('RSM reduced unseen (LOCO)', 's'), ('ANN unseen (LOCO)', '^'), (f'{BEST} unseen (LOCO)', 'o')]:
        ax.scatter(s.actual, s[col], marker=mk, alpha=.75, label=col)
    lo, hi = s.actual.min() - 3, s.actual.max() + 3
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1); ax.set(title=m, xlabel='Actual qe (mg/g)', ylabel='Predicted qe (mg/g)')
axs[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(f'{OUT}/actual_vs_predicted_v3.png', dpi=200); plt.show()

# Best model at the v1 lab validation settings (RSM optimum, pH 4-8, rounded); best model trained on all data
vp = pd.DataFrame({'mat': ['Repyrolysis 1:1', 'Repyrolysis 2:1', 'BC:MgCO3 1:1', 'BC:MgCO3 2:1'],
                   'conc': [30, 30, 65, 100], 'time': [160, 150, 190, 230], 'pH': [4.5, 5.0, 8.0, 8.0], 'dose': [15, 10, 10, 15],
                   'RSM qe (v1)': [61.39, 65.21, 113.78, 54.73]})
for k, col in zip(X4, ['conc', 'time', 'pH', 'dose']): vp[k] = to_coded(k, vp[col])
vp['logR'], vp['cond'] = np.log(vp.conc / vp.dose), -1
vp[f'{BEST} qe'] = run_candidates(d, vp, [BEST])[BEST].values.round(2)
validation = vp[['mat', 'conc', 'time', 'pH', 'dose', 'RSM qe (v1)', f'{BEST} qe']]
validation

## Export

In [ ]:
with pd.ExcelWriter(f'{OUT}/RSM_ANN_v3_results.xlsx') as xw:
    summary.to_excel(xw, sheet_name='Model selection summary')
    per_material.to_excel(xw, sheet_name='Best vs baselines per material')
    per_rep.to_excel(xw, sheet_name='5-fold CV per repeat', index=False)
    t_all.to_excel(xw, sheet_name='Hold-out all models', index=False)
    l_conf.to_excel(xw, sheet_name='LOCO confirm', index=False)
    convergence.to_excel(xw, sheet_name='ANN convergence', index=False)
    avp.to_excel(xw, sheet_name='Actual vs predicted', index=False)
    validation.to_excel(xw, sheet_name='Validation predictions', index=False)
    pd.DataFrame({'Best model': [BEST], 'Folds': [len(FOLDS)], 'Repeats': [REPEATS]}).to_excel(xw, sheet_name='Info', index=False)
with zipfile.ZipFile('RSM_ANN_v3_results.zip', 'w') as z:
    for f in os.listdir(OUT): z.write(f'{OUT}/{f}')
print(sorted(os.listdir(OUT)))
try:
    from google.colab import files; files.download('RSM_ANN_v3_results.zip')
except ImportError:
    pass